In [ ]:
from datetime import datetime

import numpy as np
from IPython.display import Audio, display
from stonesoup.types.groundtruth import GroundTruthState

from nereus.plotter import plot_spectrogram
from nereus.signal.ambient import WhiteNoise
from nereus.signal.anthropogenic import TonalSignal
from nereus.signal.biological import PointSourceSnappingShrimpSignal, WhaleCallSignal
from nereus.signal.effects import Reverb

# --- Configuration Constants ---
SAMPLING_RATE = 48000  # Sampling Rate
SIGNAL_DURATION = 10.0  # Signal duration in seconds

# Whale Calls

This section simulates the complex vocalisations of a Humpback Whale (Megaptera novaeangliae). Humpback songs are hierarchical, consisting of themes (repeating patterns) that form phrases, which in turn create a full song.

- Frequency: The fundamental frequency ranges from 250 Hz to 800 Hz, which is typical for social sounds and songs.

- Harmonics: The calls are rich in harmonics (overtones), giving them a full, trumpet-like timbre.

- Structure: The code generates a specific "song" sequence (A-B-C structure) where the whale switches between different frequency contours (flat, rising, falling).

- Effect: A reverb effect is applied to simulate the echoic nature of the underwater environment.

In [ ]:
whale_source = GroundTruthState(
    [0, 0, 0, 0],
    timestamp=datetime.now(),
    metadata={
        "amplitude_upa": 10 ** (180 / 20),
    },
)

# A theme is a list of frequency steps (Hz) from the previous point
theme_0 = [15, -15, 15, -15]  # Slight warble
theme_1 = [10, 0]  # Slight rise then flat
theme_2 = [-20, -10, 10]  # Sharp fall then slight rise
theme_3 = [0, 0, 0, 0]  # Flat theme

# A phrase is a list of theme indices
phrase_A = [0, 1]  # Sings theme 0, then theme 1
phrase_B = [2, 1]  # Sings theme 2, then theme 1
phrase_C = [3]  # Sings theme 3 (flat)

# The full song is a sequence of phrases
song_phrases = [phrase_C, phrase_A, phrase_B]  # C-A-B structure

reverb_effect = Reverb(duration_s=0.4, wet_dry_mix=0.8)

whale_signal_model = WhaleCallSignal(
    # --- General ---
    duration_s=SIGNAL_DURATION,
    sampling_rate_hz=SAMPLING_RATE,
    # --- Song Structure ---
    song_structure_enabled=True,
    song_themes=[theme_0, theme_1, theme_2, theme_3],
    song_phrases=song_phrases,
    theme_base_freq_hz=250,
    theme_freq_jitter_hz=10,
    theme_duration_s=1.2,
    # --- Call Timing & Phrasing ---
    mean_call_interval_s=2.0,
    interval_jitter_s=0.3,
    call_duration_s=1.0,
    duration_jitter_s=0.3,
    # --- Dynamic Frequency Contour ---
    start_freq_hz=250,
    start_freq_jitter_hz=20,
    end_freq_hz=800,
    end_freq_jitter_hz=50,
    num_contour_points=3,
    contour_variability_hz=10,
    # --- Timbre and Texture ---
    min_harmonics=20,
    max_harmonics=25,
    harmonic_decay_db=1,
    vibrato_rate_hz=2.5,
    vibrato_depth_hz=1.0,
    # --- Sub-harmonics ---
    sub_harmonic_ratios=[0.5],  # Add sub-harmonics at 1/2 the fundamental frequency
    sub_harmonic_amplitude_ratio=0.25,  # Volume of the sub-harmonic (25% of fundamental)
    add_breathy_noise=True,
    breathy_noise_amount=0.15,  # Mix 15% noise for a breathy quality
    breathy_noise_lp_cutoff_hz=1800,  # Filter noise to sound like air
    # --- Filtering, Envelope, and Reverb ---
    low_cutoff_hz=200,
    high_cutoff_hz=5000,
    envelope_taper_ratio=0.8,  # Smoother onset and decay
    # --- Post Processing Effects ---
    effects=[reverb_effect],
)

whale_calls_complex = whale_signal_model.generate(
    source=whale_source,
    sensor_delays_s=np.array([0.0]),
    tloss_db=90.0,
    propagation_time_s=0.0,
)
whale_calls_real = np.real(whale_calls_complex[0, :])

In [ ]:
# Spectrogram Plot
plot_spectrogram(
    whale_calls_real, SAMPLING_RATE, n_fft=4096, hop_length=1024, y_lim=(0, 5500)
).show()

# Audio
display(Audio(data=whale_calls_real, rate=SAMPLING_RATE))

# Snapping Shrimp

This section models the ambient "crackle" of a Snapping Shrimp (Alpheus heterochaelis) colony. This sound is ubiquitous in shallow tropical and subtropical waters and is often described as sounding like "frying bacon."

- Mechanism: The sound is not created by the shells hitting together, but by a cavitation bubble collapsing. The shrimp closes its snapper claw so fast (over 100 km/h) that it creates a low-pressure bubble that violently collapses, emitting a loud "snap" and a flash of light (sonoluminescence).

- Spectrum: The signal is broadband, covering a wide frequency range from a few kHz up to 200 kHz.

- Distribution: We simulate a "point-source" to represent a colony in the far-field. A "diffuse" model is also available for a near-field source.

In [ ]:
shrimp_source = GroundTruthState(
    [0, 0, 0, 0, -50, 0],
    timestamp=datetime.now(),
    metadata={
        "amplitude_upa": 10 ** (195 / 20),
        "position_mapping": [0, 2, 4],
    },
)

effects = [Reverb(duration_s=0.1, wet_dry_mix=0.9)]

shrimp_signal_model = PointSourceSnappingShrimpSignal(
    duration_s=SIGNAL_DURATION,
    sampling_rate_hz=SAMPLING_RATE,
    temperature_celsius=25,
    start_time_hours=18.0,
    diurnal_amplitude=0.25,
    diurnal_phase_hours=6,
    delay_duration=0.0006,
    onset_duration=0.0001,
    snap_duration=0.0014,
    onset_level=0.15,
    onset_freq=2500,
    snap_decay=1000,
    low_cutoff_hz=1500,
    high_cutoff_hz=10000,
)

shrimp_signal = shrimp_signal_model.generate(
    source=shrimp_source,
    sensor_delays_s=np.array([0.0]),
    tloss_db=80.0,
    propagation_time_s=10.0,
)
shrimp_signal_real = np.real(shrimp_signal[0, :])

In [ ]:
# Spectrogram Plot
plot_spectrogram(
    shrimp_signal_real, SAMPLING_RATE, n_fft=2048, hop_length=512, y_lim=(0, 20000)
).show()

# Audio
display(Audio(data=shrimp_signal_real, rate=SAMPLING_RATE))

# Commerical Vessel

This section generates the low-frequency noise characteristic of a large commercial ship. Ship noise is the dominant source of anthropogenic (human-made) sound in the ocean.

- Tonal Components: The signal consists of discrete spectral lines (tonals) at low frequencies (50, 75, 82, 125 Hz).

- Origin: These tones are mechanically generated by the rotation of the ship's propeller and engine machinery.

    - Blade Rate: The frequency at which the propeller blades pass a specific point (Shaft RPM × Number of Blades).

    - Machinery: Radiated noise from the engines and generators.

- Continuous Nature: Unlike the transient whale calls or shrimp snaps, ship noise is continuous and stable over short durations.

In [ ]:
target_states = [
    GroundTruthState(
        [0, 0, 0, 0, -10, 0],
        timestamp=datetime.now(),
        metadata={
            "frequencies_hz": np.array(
                [
                    # --- Low-Frequency Prop/Engine Tonals ---
                    # Urick, R. J. (1983). Principles of underwater sound.
                    50.0,  # 4th Blade Rate Harmonic + Main Machinery
                    75.0,  # 6th Blade Rate Harmonic
                    125.0,  # 10th Blade Rate Harmonic
                    82.0,  # Unrelated secondary machinery
                ]
            ),
            # Amplitudes (Source Levels) in dB re 1uPa @ 1m.
            "amplitudes_upa": 10
            ** (
                np.array(
                    [
                        175.0,
                        168.0,
                        162.0,
                        160.0,
                    ]
                )
                / 20
            ),
            "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
            "position_mapping": [0, 2, 4],
        },
    )
]

tonal_signal = TonalSignal(
    duration_s=SIGNAL_DURATION,
    sampling_rate_hz=SAMPLING_RATE,
).generate(
    source=target_states[0], sensor_delays_s=np.array([0.0]), tloss_db=60.0, propagation_time_s=0.0
)

tonal_signal_real = np.real(tonal_signal[0, :])

In [ ]:
# Spectrogram Plot
plot_spectrogram(
    tonal_signal_real,
    SAMPLING_RATE,
    n_fft=4096 * 6,
    hop_length=1024,
    y_lim=(0, 200),
    yaxis_format="hz",
).show()

# Audio
display(Audio(data=tonal_signal_real, rate=SAMPLING_RATE))

# Ambient Background Noise

This section generates a background White Noise baseline. In a real ocean environment, "ambient noise" is the composite sound of wind, waves, rain, and distant shipping that cannot be resolved into individual sources.

- Statistical Model: We use a white (Gaussian) noise process here for simplicity. This assumes energy is equally distributed across all frequencies.

- Wenz Curves: In more complex models, the intensity would vary by frequency according to the "Wenz Curves," where low frequencies are dominated by shipping and high frequencies by wind and rain.

In [ ]:
ambient_noise = WhiteNoise(
    amplitude_upa=10 ** (90 / 20), duration_s=SIGNAL_DURATION, sampling_rate_hz=SAMPLING_RATE
).generate()

ambient_noise_real = np.real(ambient_noise[0, :])

In [ ]:
# Spectrogram Plot
plot_spectrogram(
    ambient_noise_real,
    SAMPLING_RATE,
    n_fft=4096,
    hop_length=1024,
    y_lim=(0, 20000),
    yaxis_format="kHz",
).show()

# Audio
display(Audio(data=ambient_noise_real, rate=SAMPLING_RATE))

# Soundscape

This final section combines all the individual signals into a single composite soundscape. This represents what a hydrophone (underwater microphone) would actually record in a busy shallow-water environment.

You can see (and hear) how the signals overlap. The low-frequency ship noise masks the lower harmonics of the whale call, while the broadband shrimp noise raises the noise floor across the entire spectrum.

Spectrogram Analysis:

- Bottom (< 200 Hz): Dominated by the constant horizontal lines of the ship.

- Middle (200 - 5000 Hz): The sweeping contours of the whale calls are clearly visible.

- Background: The "speckled" texture covering the whole plot is the shrimp and ambient noise.

In [ ]:
# Add all signals together
soundscape = whale_calls_real + shrimp_signal_real + tonal_signal_real + ambient_noise_real

# Plot Spectrogram
plot_spectrogram(soundscape, SAMPLING_RATE, n_fft=4096, hop_length=1024, y_lim=(0, 5500)).show()

# Display Audio Player
display(Audio(data=soundscape, rate=SAMPLING_RATE))